# 02 - Cleaning and Feature Engineering

This notebook prepares a modeling-ready dataset. The cleaning strategy keeps the audio and business-relevant columns, removes clear identifiers from model features, imputes missing values, creates useful engineered variables, and saves the final clean dataset for the remaining notebooks.

In [1]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw" / "dataset.csv"
DATA_CLEAN = PROJECT_ROOT / "data" / "processed" / "spotify_tracks_clean.csv"

RANDOM_STATE = 42

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
df = pd.read_csv(DATA_RAW)
print(df.shape)
display(df.head())

(114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## Cleaning Decisions

`Unnamed: 0` is an index artifact, so it is removed. Song identifiers and text metadata are useful for interpretation but are not predictive audio features. For modeling fairness and leakage prevention, `track_id`, `track_name`, `album_name`, and `artists` are excluded from model feature matrices.

In [3]:
df_clean = df.copy()
if "Unnamed: 0" in df_clean.columns:
    df_clean = df_clean.drop(columns=["Unnamed: 0"])

before_rows = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["track_id"]).reset_index(drop=True)
print(f"Rows before duplicate removal: {before_rows:,}")
print(f"Rows after duplicate track_id removal: {len(df_clean):,}")

Rows before duplicate removal: 114,000
Rows after duplicate track_id removal: 89,741


## Missing Data Imputation

Numeric columns are imputed with medians because medians are robust to outliers. Categorical columns are imputed with the mode or `Unknown` when a mode is unavailable.

In [4]:
numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df_clean.select_dtypes(exclude=np.number).columns.tolist()

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    mode = df_clean[col].mode(dropna=True)
    fill_value = mode.iloc[0] if not mode.empty else "Unknown"
    df_clean[col] = df_clean[col].fillna(fill_value)

missing_after = df_clean.isna().sum().sum()
print(f"Remaining missing values: {missing_after}")

Remaining missing values: 0


## Feature Engineering

The engineered features improve interpretability and make later notebooks simpler:

- `duration_min` converts milliseconds to minutes.
- `is_explicit` converts the Boolean explicit flag into 0/1.
- `popularity_class` creates balanced Low/Medium/High labels using quantile bins for classification.

In [5]:
df_clean["duration_min"] = df_clean["duration_ms"] / 60000
df_clean["is_explicit"] = df_clean["explicit"].astype(int)

labels = ["Low", "Medium", "High"]
df_clean["popularity_class"] = pd.qcut(
    df_clean["popularity"],
    q=3,
    labels=labels,
    duplicates="drop",
)

display(df_clean[["popularity", "duration_min", "is_explicit", "popularity_class"]].head())
display(df_clean["popularity_class"].value_counts().to_frame("count"))

,popularity,duration_min,is_explicit,popularity_class
0,73,3.844433,0,High
1,55,2.493500,0,High
2,57,3.513767,0,High
3,71,3.365550,0,High
4,82,3.314217,0,High


,count
popularity_class,
Low,31659
High,29870
Medium,28212


## Scaling Preview

Scaling is not saved over the original values because train/test splits must fit scalers only on training data to avoid leakage. This preview shows why scaling will be applied inside each modeling notebook.

In [6]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

audio_features = [
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "duration_min",
    "key", "mode", "time_signature", "is_explicit"
]

standard_preview = pd.DataFrame(
    StandardScaler().fit_transform(df_clean[audio_features]),
    columns=audio_features
).describe().loc[["mean", "std"]]

normalized_preview = pd.DataFrame(
    MinMaxScaler().fit_transform(df_clean[audio_features]),
    columns=audio_features
).describe().loc[["min", "max"]]

display(standard_preview.round(3))
display(normalized_preview.round(3))

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_min,key,mode,time_signature,is_explicit
mean,-0.0,0.0,-0.0,0.0,0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_min,key,mode,time_signature,is_explicit
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## One-Hot Encoding Preview

`track_genre` is categorical. It is kept in the clean dataset and one-hot encoded inside modeling pipelines so the encoding is reproducible and can be fitted only on training data.

In [7]:
genre_encoded_preview = pd.get_dummies(df_clean[["track_genre"]], drop_first=True)
print(f"Genre dummy columns created: {genre_encoded_preview.shape[1]:,}")
display(genre_encoded_preview.head())

Genre dummy columns created: 112


,track_genre_afrobeat,track_genre_alt-rock,track_genre_alternative,track_genre_ambient,track_genre_anime,track_genre_black-metal,track_genre_bluegrass,track_genre_blues,track_genre_brazil,track_genre_breakbeat,...,track_genre_spanish,track_genre_study,track_genre_swedish,track_genre_synth-pop,track_genre_tango,track_genre_techno,track_genre_trance,track_genre_trip-hop,track_genre_turkish,track_genre_world-music
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [8]:
DATA_CLEAN.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(DATA_CLEAN, index=False)
print(f"Saved clean dataset to: {DATA_CLEAN}")
print(f"Clean shape: {df_clean.shape}")

Saved clean dataset to: C:\Users\T480\Desktop\tec\8vo\compiladores\ia\proyecto-ia-tc3002b\data\processed\spotify_tracks_clean.csv
Clean shape: (89741, 23)


## Cleaning Summary

The final clean dataset removes duplicate track IDs, fills missing values, adds classification and duration features, and preserves raw audio feature values so each algorithm can apply scaling correctly after its train/test split.